In [0]:
catalog='workspace'
schema='practise_db'
volume_path=f"/Volumes/{catalog}/{schema}/staging"
source_file=f"{volume_path}/customer_source.csv"

# Table Names for each medalian layer

brz_table=f"{catalog}.{schema}.brz_customers"
slv_table=f"{catalog}.{schema}.slv_customers"
gld_table=f"{catalog}.{schema}.gld_customers"


In [0]:
import os 

if not os.path.exists(source_file):
    raise FileNotFoundError(
                f"Source File Missing: '{source_file}' was not found."
                f"Please Ensure the source file uploaded to the staging volume"
                f"before running the pipeline"
        )

if os.path.getsize(source_file)==0:
    raise ValueError(
                f"source file empty : '{source_file}' exists but is 0 bytes"
                f"The source file must contain data."
        )
    
print(f"Source File Validation passed: {source_file} ({os.path.getsize(source_file)} bytes)")


In [0]:
brz_df=(
    spark.read.format('csv').option('header',True).option('inferSchema',True).load(source_file)
)

(
    brz_df.write.mode('overwrite').format('delta').saveAsTable(brz_table)
)

print(f"Bronze Layer Loaded : {brz_table} ({brz_df.count()} rows)")

display(spark.table(brz_table))

In [0]:
from pyspark.sql.functions import col,trim,to_date,regexp_replace 

slv_df=(
        spark.table(brz_table)
        #Rename Columns to snake_case / Standardized Naming 
        .withColumnRenamed('cust_id','customer_id')
        .withColumnRenamed('full_name','customer_name')
        .withColumnRenamed('signup_date','signup_date_str')
        # Clean String Columns (trim,whitespace,lowercase email)
        .withColumn('customer_name',trim(col('customer_name')))
        .withColumn('email',trim(col('email')))
        .withColumn('city',trim(col('city')))
        # Convert date string to proper data types
        .withColumn('signup_date',to_date(col('signup_date_str')))
        .withColumn('order_total',col('order_total').cast("decimal(10,2)"))
        # Drop the Intermediate string column 
        .drop('signup_date_str')
        #Data Quality : Filter out with null customer_id 
        .filter(col('customer_id').isNotNull())

)

(
    slv_df.write.mode('overwrite').format('delta').saveAsTable(slv_table)
)


print(f"Silver Layer Loaded: {slv_table} ({slv_df.count()}) rows")

display(spark.table(slv_table))

In [0]:
from pyspark.sql.functions import count,sum as _sum, round as _round 

gold_df=(
            spark.table(slv_table)
            .groupBy('city')
            .agg(
                count('customer_id').alias('customer_count'),
                _sum('order_total').alias('total_revenue'),
                _round(_sum('order_total')/count('customer_id'),2).alias('avg_order_value')
            )
            .orderBy(col('total_revenue').desc())
)

(gold_df.write.mode('overwrite').format('delta').saveAsTable(gld_table))

print(f"Gold Layer Loaded: {gld_table} ({gold_df.count()}) rows")

display(spark.table(gld_table))

In [0]:
import os

if os.path.exists(source_file):
    os.remove(source_file)
    print(f"Removed Source File : {source_file}")
else:
    print(f"Source File already removed: {source_file}")